# Bài 13 · Trực quan hoá nâng cao và phản biện biểu đồ

**Lập trình xử lý dữ liệu · 2627-1 · Viện TTNT, UET-VNU**

Chọn **Save a copy in Drive** trước khi sửa.

## Mục tiêu bài học

1. So sánh phân phối giữa các nhóm bằng seaborn: boxplot, `hue`, `col`.
2. Chọn cách biểu diễn (độ dài, góc, diện tích, màu) và bảng màu theo loại dữ liệu.
3. Phản biện và sửa ba lỗi trình bày thường gặp bằng năm điểm kiểm tra của Bài 12.
4. Tự học: bản đồ theo quận (choropleth) và biểu đồ tương tác.

**Giáo trình:** [McKinney, chương 9, §9.2](https://wesmckinney.com/book/plotting-and-visualization).
Dữ liệu: Santiago, mốc chụp **29/06/2026**, các file `visualisations/` (giá là **CLP/đêm**).
Các ví dụ được kiểm tra bằng pandas **3.0.3**, seaborn **0.13** và geopandas **1.1**.

## Chuẩn bị

Ô cài đặt dành cho Colab. Nếu phiên Colab đã nạp pandas phiên bản khác, chọn
**Restart session and run all** sau khi cài.
Chạy các ô demo theo thứ tự; ô **Bài tập** để bạn tự viết lời giải và nhận xét.
Ô dưới dùng file đã tải nếu có; nếu chưa có thì đọc đúng mốc chụp từ nguồn (khoảng 20 MB).

In [ ]:
%pip -q install pandas==3.0.3 "seaborn>=0.13" geopandas

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE = ("https://data.insideairbnb.com/chile/rm/santiago/"
        "2026-06-29/visualisations")


def doc(ten_tep, ten_nguon):
    """Đường dẫn file đã tải trong data/ nếu có, nếu không thì URL nguồn."""
    tep = Path("data") / ten_tep
    return tep if tep.exists() else f"{BASE}/{ten_nguon}"


df = pd.read_csv(doc("listings_santiago_2026-06-29.csv", "listings.csv"))
rv = pd.read_csv(doc("reviews_santiago_2026-06-29.csv", "reviews.csv"),
                 parse_dates=["date"])
co_gia = df.dropna(subset=["price"])          # chỗ ở có giá

XANH, CAM = "#1E93AB", "#E8890C"
print(df.shape, co_gia.shape, rv.shape)
print("pandas:", pd.__version__)

## 1. So sánh phân phối giữa các nhóm

seaborn xây trên matplotlib: nhận DataFrame và tên cột, tự tách nhóm, tự đặt nhãn từ tên cột.

In [ ]:
import seaborn as sns
sns.set_theme(style="whitegrid")    # phong cách chung

co_gia = df.dropna(subset=["price"])     # chỗ ở có giá
ax = sns.histplot(data=co_gia, x="price", log_scale=True)
ax.set_xlabel("giá (CLP/đêm, thang log)")
plt.show()

Một boxplot tóm tắt một phân phối bằng năm giá trị: hộp từ Q1 đến Q3 (chứa 50% ở giữa, độ dài là IQR),
vạch giữa hộp là trung vị, râu kéo tới điểm xa nhất còn trong 1,5 × IQR, ngoài nữa là điểm ngoại lai.

In [ ]:
pr = co_gia.loc[co_gia["room_type"] == "Private room", "price"]
q1, tv, q3 = pr.quantile([0.25, 0.5, 0.75])
print(f"phòng riêng: n = {len(pr)}, Q1 = {q1:.0f}, trung vị = {tv:.0f}, Q3 = {q3:.0f}")

Boxplot theo loại phòng, thang log, ẩn điểm ngoại lai:

In [ ]:
thu_tu = ["Shared room", "Private room",
          "Entire home/apt", "Hotel room"]
ax = sns.boxplot(data=co_gia, x="price", y="room_type",
                 order=thu_tu, log_scale=True, showfliers=False, color=XANH)
ax.set_xlabel("giá (CLP/đêm, thang log)")
ax.set_ylabel("")
plt.show()
print(co_gia.groupby("room_type")["price"].agg(["median", "size"]))

`hue=` tách nhóm bằng màu; `stat="density", common_norm=False` chuẩn hoá riêng từng nhóm để so sánh hình dạng.

In [ ]:
hai_quan = co_gia[co_gia["neighbourhood"].isin(["Santiago", "Las Condes"])]
ax = sns.histplot(data=hai_quan, x="price", hue="neighbourhood",
                  log_scale=True, stat="density", common_norm=False,
                  element="step",                                # chỉ vẽ đường viền, dễ nhìn khi chồng
                  palette={"Santiago": XANH, "Las Condes": CAM})
ax.set_xlabel("giá (CLP/đêm, thang log)")
ax.set_title("Giá trung vị ở Las Condes gấp khoảng hai lần quận Santiago",
             loc="left", fontweight="bold")
plt.show()
print(hai_quan.groupby("neighbourhood")["price"].agg(["median", "size"]))

`col=` (trong `displot`) vẽ mỗi nhóm một khung nhỏ, chung thang trục; `hue=` chồng các nhóm lên cùng một khung.

In [ ]:
bon_quan = ["Santiago", "Providencia", "Las Condes", "Ñuñoa"]
g = sns.displot(data=co_gia[co_gia["neighbourhood"].isin(bon_quan)],
                x="price", col="neighbourhood", col_order=bon_quan, log_scale=True,
                stat="density", common_norm=False, height=2.8, color=XANH)
g.set(xlim=(5e3, 2e6))                       # displot trả về FacetGrid: chỉnh cả lưới một lần
g.set_titles("{col_name}")
plt.show()

Heatmap (bản đồ nhiệt) tô màu từng ô của bảng pivot (Bài 5) và có thể ghi giá trị trong ô:

In [ ]:
quan_dong = df["neighbourhood"].value_counts().head(8).index
pv = (co_gia[co_gia["neighbourhood"].isin(quan_dong)]
      .pivot_table(values="price", index="neighbourhood",
                   columns="room_type", aggfunc="median")
      [["Entire home/apt", "Private room"]] / 1000)
print(pv.round(1))
ax = sns.heatmap(pv.sort_values("Entire home/apt", ascending=False),
                 annot=True, fmt=".0f", cmap="Blues",
                 cbar_kws={"label": "giá trung vị (nghìn CLP/đêm)"})
ax.set_xlabel(""); ax.set_ylabel("")
plt.show()

### Bài tập 1: Vẽ theo số đếm hay mật độ?

Vẽ lại hình của hai quận ở trên, bỏ `stat` và `common_norm` (mặc định là đếm số chỗ ở).

**Dự đoán trước khi chạy: hai nhóm trông thế nào? Mỗi hình trả lời câu hỏi nào?**

In [ ]:
# Bài tập 1: viết lời giải ở đây

## 2. Chọn cách biểu diễn dữ liệu

Năm quận đông nhất, mã hoá bằng độ dài, góc, diện tích, màu. Thử đọc B hay C lớn hơn trên từng hình, rồi so với số thật:

In [ ]:
nam_quan = df["neighbourhood"].value_counts().head(5)
print(nam_quan)

Chọn bảng màu theo loại dữ liệu: **tuần tự** cho độ lớn, **phân kỳ** cho độ lệch hai phía quanh một giá trị giữa,
**định tính** cho nhóm không có thứ tự. Tránh `jet`: độ sáng lên xuống làm mắt thấy ranh giới không có thật.

## 3. Phản biện ba biểu đồ lỗi

Ba hình dưới có lỗi trình bày thường gặp trong hình do AI sinh. Với mỗi hình, ghi lại: (a) hình gợi ra kết luận gì,
(b) lựa chọn trình bày nào tạo ra ấn tượng đó, (c) điểm kiểm tra nào của Bài 12 phát hiện được lỗi.

### Phản biện hình A

In [ ]:
# Hình lỗi A. Số đánh giá là dữ liệu thật; chuỗi giá là số MINH HOẠ tự sinh.
thang = (rv[rv["date"] < "2026-06-01"]
         .set_index("date").resample("ME").size().loc["2024":])
gia_minh_hoa = pd.Series(np.linspace(55, 62, len(thang))
                         + np.random.default_rng(3).normal(0, 1.2, len(thang)),
                         index=thang.index)

fig, ax1 = plt.subplots(figsize=(8.6, 3.6))
ax1.plot(thang.index, thang.values, color=XANH, lw=2)
ax1.set_ylabel("số đánh giá / tháng", color=XANH)
ax2 = ax1.twinx()                          # trục y thứ hai, dùng chung trục x
ax2.plot(gia_minh_hoa.index, gia_minh_hoa.values, color="red", ls="--", lw=2)
ax2.set_ylabel("giá trung vị (nghìn CLP, minh hoạ)", color="red")
ax2.set_ylim(52, 64)
ax1.set_title("Nhu cầu tăng làm giá tăng", fontweight="bold")
plt.show()

dau, cuoi = gia_minh_hoa.iloc[:3].mean(), gia_minh_hoa.iloc[-3:].mean()
print(f"số đánh giá: {thang.iloc[0]} -> {thang.iloc[-1]} (gấp {thang.iloc[-1] / thang.iloc[0]:.1f} lần)")
print(f"giá minh hoạ, TB 3 tháng đầu -> 3 tháng cuối: {dau:.1f} -> {cuoi:.1f} (+{cuoi / dau - 1:.0%})")

**Sửa hình A:** tách thành hai khung chung trục thời gian (`plt.subplots(2, 1, sharex=True)`), trục y từ 0, tiêu đề không khẳng định quan hệ.

In [ ]:
fig, (axA, axB) = plt.subplots(2, 1, sharex=True, figsize=(8.6, 4))
axA.plot(thang.index, thang.values, color=XANH, lw=2)
axA.set_ylabel("số đánh giá\n/ tháng")
axA.set_ylim(0, None)
axB.plot(gia_minh_hoa.index, gia_minh_hoa.values, color=CAM, lw=2)
axB.set_ylabel("giá trung vị\n(nghìn CLP)")
axB.set_ylim(0, 70)
axA.set_title("Số đánh giá và giá cùng tăng theo thời gian",
              loc="left", fontweight="bold")
plt.show()

### Phản biện hình B

In [ ]:
# Hình lỗi B: biểu đồ tròn 12 lát (dữ liệu thật)
quan12 = df["neighbourhood"].value_counts().head(12)
fig, ax = plt.subplots(figsize=(6.6, 4.4))
ax.pie(quan12.values, labels=quan12.index, colors=plt.cm.tab20.colors,
       startangle=90, textprops={"fontsize": 8})
ax.set_title("Phân bố chỗ ở theo quận", fontweight="bold")
plt.show()
print(f"12 quận chiếm {quan12.sum() / len(df):.1%} số chỗ ở")

**Sửa hình B:** thanh ngang xếp hạng theo % tổng số chỗ ở, thêm một thanh cho các quận còn lại.

In [ ]:
pct = df["neighbourhood"].value_counts(normalize=True) * 100
K = 7                                                     # 7 quận đông nhất + phần còn lại
con_lai = pd.Series({f"{len(pct) - K} quận còn lại": pct.iloc[K:].sum()})
bang_b = pd.concat([pct.iloc[:K], con_lai]).iloc[::-1]    # đảo để thanh lớn nhất ở trên
mau = ["#9a9a9a" if "còn lại" in q else (CAM if q == "Santiago" else XANH) for q in bang_b.index]

fig, ax = plt.subplots(figsize=(8, 3.6))
bars = ax.barh(bang_b.index, bang_b.values, color=mau, height=0.7)
ax.bar_label(bars, [f" {v:.1f}%".replace(".", ",") for v in bang_b.values])
ax.set_xlabel(f"% tổng số chỗ ở (n = {len(df):,})".replace(",", "."))
ax.set_title("Quận Santiago chiếm 38,8% số chỗ ở", loc="left", fontweight="bold")
plt.show()

### Phản biện hình C

In [ ]:
# Hình lỗi C: số đánh giá mỗi năm của hai quận (dữ liệu thật)
dg = rv.merge(df[["id", "neighbourhood"]], left_on="listing_id", right_on="id")
dg = dg[(dg["date"] >= "2022-01-01") & (dg["date"] < "2026-01-01")]   # bỏ năm 2026 chưa trọn
nam = dg.groupby([dg["date"].dt.year, "neighbourhood"]).size().unstack()
hai = nam[["Santiago", "Estación Central"]]
print(hai)

ax = hai.plot(marker="o", figsize=(8.4, 3.4))
ax.set_xticks(hai.index)
ax.set_ylabel("số đánh giá / năm")
ax.set_title("Số đánh giá ở Estación Central gần như không tăng", fontweight="bold")
plt.show()

**Sửa hình C:** quy hai chuỗi về chỉ số, kỳ gốc 2022 = 100. Cùng đơn vị thì so sánh được nhịp tăng.

In [ ]:
chi_so = hai / hai.loc[2022] * 100
print(chi_so.round(0))

ax = chi_so.plot(marker="o", figsize=(8.4, 3.4))
ax.axhline(100, color="#777", lw=1, ls="--")
ax.set_xticks(chi_so.index)
ax.set_ylabel("chỉ số (2022 = 100)")
ax.set_title("Estación Central tăng nhanh hơn Santiago",
             loc="left", fontweight="bold")
plt.show()

Ba lỗi, ba điểm kiểm tra:

| Hình | Lỗi | Điểm kiểm tra | Sửa |
|---|---|---|---|
| A | trục kép, tiêu đề nhân quả | trục; nguồn và diễn giải | hai khung chung trục x |
| B | góc khó so sánh, bỏ phần còn lại | dạng và màu; nguồn và diễn giải | thanh ngang, đủ 100% |
| C | hai quy mô trên một trục | trục; nguồn và diễn giải | chỉ số, kỳ gốc = 100 |

### Bài tập 2: Lập hồ sơ lỗi

**Áp dụng năm điểm kiểm tra cho hình dưới. Hình thiếu gì? Sửa thế nào?**

In [ ]:
tk = co_gia.groupby("neighbourhood")["price"].agg(median="median", n="size")
top8 = tk.nlargest(8, "median").sort_values("median")

fig, ax = plt.subplots(figsize=(8, 3.6))
bars = ax.barh(top8.index, top8["median"] / 1000, color=XANH, height=0.6)
ax.bar_label(bars, [f" {v / 1000:.0f}" for v in top8["median"]], fontsize=9)   # trục đã là nghìn CLP
ax.set_xlabel("giá trung vị (nghìn CLP/đêm)")
ax.set_title("La Pintana là quận đắt thứ hai ở Santiago", loc="left", fontweight="bold")
plt.show()

In [ ]:
# Bài tập 2: viết lời giải ở đây

## 4. Tự học: bản đồ và biểu đồ tương tác

**Choropleth** (bản đồ tô màu theo vùng): đọc ranh giới quận (GeoJSON), tính giá trung vị theo quận, `merge` rồi
`plot(column=...)`. Quận quá ít chỗ ở thì không tô màu, vì trung vị của 1 chỗ ở không đại diện cho quận.

In [ ]:
import geopandas as gpd

geo = gpd.read_file(doc("neighbourhoods_santiago_2026-06-29.geojson",
                        "neighbourhoods.geojson"))
print(geo.shape)
print(geo.head(3))

kpi = (co_gia.groupby("neighbourhood")
       .agg(gia=("price", "median"), n=("price", "size"))
       .reset_index())
ban_do = geo.merge(kpi, on="neighbourhood", how="left")   # giữ đủ 32 quận
ban_do["gia_hien"] = ban_do["gia"].where(ban_do["n"] >= 30)   # dưới 30 chỗ ở: không tô

ax = ban_do.plot(column="gia_hien", cmap="Blues", legend=True,
                 edgecolor="#999", linewidth=0.5,
                 missing_kwds={"color": "#eeeeee", "hatch": "///"},
                 legend_kwds={"label": "giá trung vị (CLP/đêm)"})
ax.set_axis_off()
plt.show()
print(ban_do.sort_values("gia", ascending=False)[["neighbourhood", "gia", "n"]].head(5))

plotly express vẽ biểu đồ tương tác: rê chuột lên điểm để xem chi tiết. Chạy ô dưới rồi thử rê chuột:

In [ ]:
import plotly.express as px

mau = co_gia.sample(3000, random_state=1)     # lấy mẫu cho nhẹ trình duyệt
fig = px.scatter(mau, x="longitude", y="latitude", color="room_type",
                 hover_name="name", hover_data=["price", "neighbourhood"],
                 opacity=0.6, height=450)
fig.show()

### Đọc thêm

1. [McKinney, chương 9, §9.2–9.3](https://wesmckinney.com/book/plotting-and-visualization)
2. [Berkeley Data 100: Visualization II](https://ds100.org/course-notes/visualization-2)
3. [seaborn: User guide and tutorial](https://seaborn.pydata.org/tutorial.html)
4. [geopandas: Mapping and plotting](https://geopandas.org/en/stable/docs/user_guide/mapping.html), [plotly express](https://plotly.com/python/plotly-express/)
5. [Financial Times: Visual Vocabulary](https://github.com/Financial-Times/chart-doctor/tree/main/visual-vocabulary)

---

## Bài tập thêm (tự làm)

### Bài tập 3: Giá và quy mô trên hai bản đồ

Vẽ hai bản đồ cạnh nhau (`plt.subplots(1, 2)`, truyền `ax=` cho từng lần `plot`): (a) giá trung vị theo quận,
chỉ tô quận có từ 30 chỗ ở; (b) **số chỗ ở** theo quận. Viết ba câu: khu nào đắt, khu nào nhiều chỗ ở,
và diện tích các quận ảnh hưởng thế nào đến cách đọc bản đồ (a).

In [ ]:
# Bài tập 3: viết lời giải ở đây


### Luyện tập thêm với thành phố của nhóm

Tìm một biểu đồ trong bản phân tích của nhóm (hoặc nhờ AI vẽ một hình so sánh hai nhóm), lập hồ sơ lỗi theo
năm điểm kiểm tra rồi vẽ bản sửa.

---

## Tóm tắt bài học

- seaborn nhận DataFrame và tên cột; boxplot, `hue`, `col` để so sánh phân phối giữa các nhóm.
- Mắt so sánh vị trí và độ dài tốt nhất; chọn bảng màu theo loại dữ liệu.
- Ba lỗi hay gặp: trục kép, biểu đồ tròn nhiều lát, hai quy mô trên một trục; cách sửa tương ứng.
- Rà soát mọi hình bằng năm điểm: thông điệp, trục, cỡ mẫu, dạng và màu, nguồn và diễn giải.

Hoàn thành hai bài tập rồi chạy **Restart session and run all**.

**Khai báo AI (nếu dùng):** công cụ, yêu cầu chính, một kết quả bạn đã tự kiểm tra. …

**Bài sau:** kể chuyện bằng dữ liệu và thẩm định một bản phân tích do AI viết.